# 03 — DL Risk Model (Heart Disease, PyTorch MLP)

Small two-hidden-layer MLP with dropout, trained on the same preprocessed tabular features as the gradient-boosting baseline. Used as a second, structurally different scorer that the agent can compare against the ML model.

**Lineage.** Project 4 (Fashion-MNIST CNN with dropout): fixed seed 42, dropout regularization, per-epoch loss/accuracy logging, per-slice disaggregated evaluation, and CPU-budget honesty. The implementation here is a fresh tabular MLP, not a CNN.

**Educational artifact only. Not for clinical use.**

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_heart_disease
from src.preprocessing import split_and_preprocess
from src.dl_model import train_and_evaluate, save, slice_metrics, predict_proba

## 1. Train

In [ ]:
df = load_heart_disease()
split = split_and_preprocess(df)
model, metrics = train_and_evaluate(split, epochs=80)
input_dim = split.preprocessor.transform(split.X_train).shape[1]
save(model, input_dim=input_dim)
print('Test ROC-AUC:', round(metrics.test_roc_auc, 4))
print('Test PR-AUC :', round(metrics.test_pr_auc, 4))
print('Test Brier  :', round(metrics.test_brier, 4))

## 2. Training curves
Lineage: P4 reported per-epoch loss + accuracy and explicitly compared baseline vs dropout-regularized models. Here we report per-epoch train/test loss and test AUC.

In [ ]:
hist = pd.DataFrame(metrics.history)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hist['epoch'], hist['train_loss'], label='train')
axes[0].plot(hist['epoch'], hist['test_loss'], label='test', linestyle='--')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('BCE loss'); axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(hist['epoch'], hist['test_auc'], color='#5BA37F')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('Test ROC-AUC'); axes[1].set_title('Test AUC')
plt.tight_layout(); plt.show()

## 3. Per-slice metrics — bias audit
Independent model, same disaggregation. If both ML and DL show similar gaps on the same slices, that's evidence the gap is in the *data* (selection bias, small slice sizes), not in any one model's quirk.

In [ ]:
print('--- By sex (0 = female, 1 = male) ---')
print(slice_metrics(model, split.preprocessor, split.X_test, split.y_test, 'sex'))

In [ ]:
X_test_aug = split.X_test.copy()
X_test_aug['age_band'] = pd.cut(
    X_test_aug['age'], bins=[0, 45, 55, 65, 120],
    labels=['<45', '45-54', '55-64', '65+']
)
print('--- By age band ---')
print(slice_metrics(model, split.preprocessor, X_test_aug, split.y_test, 'age_band'))

## 4. Notes for the synthesis paper
- The MLP here is intentionally small (~5k parameters) and trained on CPU in seconds. It is *not* a state-of-the-art tabular model; its job is to provide a second, structurally different scorer for the ensemble layer (Phase E) and the agent orchestrator (Phase H).
- Single seed (42) — per-epoch test-AUC noise visible in the curves indicates a multi-seed average would be the right next step for any rigorous comparison.
- Same per-sex slice gap appears in both ML and DL → the disparity is data-driven, not model-driven.